In [ ]:
import torch
from torch import Tensor, nn
import numpy as np
import plotly.graph_objects as go
import einops
import itertools
import sys 
from pathlib import Path


# Make sure exercises are in the path
chapter = "chapter0_fundamentals"
section = "part2_cnns"
#root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
root_dir = Path("/Users/sebastin/Documents/perso/ARENA_training/ARENA_3.0")
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

MAIN = __name__ == "__main__"

import part2_cnns.tests as tests
import part2_cnns.utils as utils
from plotly_utils import line

In [ ]:
class ReLU(nn.Module):
    def forward(self, x: Tensor) -> Tensor:
        return torch.max(x, torch.zeros_like(x))


tests.test_relu(ReLU)

All tests in `test_relu` passed!


In [ ]:
class Linear(nn.Module):
    def __init__(self, in_features: int, out_features: int, bias=True):
        """
        A simple linear (technically, affine) transformation.

        The fields should be named `weight` and `bias` for compatibility with PyTorch.
        If `bias` is False, set `self.bias` to None.
        """
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_features, in_features))
        if bias:
            self.bias = nn.Parameter(torch.randn(out_features))
        else:
            self.bias = None

        # weight initialization
        nn.init.kaiming_uniform_(self.weight, a=0)


    def forward(self, x: Tensor) -> Tensor:
        """
        x: shape (*, in_features)
        Return: shape (*, out_features)
        """
        y = x @ self.weight.T + (self.bias if self.bias is not None else 0)
        return y

    def extra_repr(self) -> str:
        return f"in_features={self.weight.shape[1]}, out_features={self.weight.shape[0]}, bias={self.bias is not None}"
    


tests.test_linear_parameters(Linear, bias=False)
tests.test_linear_parameters(Linear, bias=True)
tests.test_linear_forward(Linear, bias=False)
tests.test_linear_forward(Linear, bias=True)

All tests in `test_linear_parameters` passed!
All tests in `test_linear_parameters` passed!
All tests in `test_linear_forward` passed!
All tests in `test_linear_forward` passed!


In [ ]:
class Flatten(nn.Module):
    def __init__(self, start_dim: int = 1, end_dim: int = -1) -> None:
        super().__init__()
        self.start_dim = start_dim
        self.end_dim = end_dim

    def forward(self, input: Tensor) -> Tensor:
        """
        Flatten out dimensions from start_dim to end_dim, inclusive of both.
        """
        shape = input.shape

        # Get start & end dims, handling negative indexing for end dim
        start_dim = self.start_dim
        end_dim = self.end_dim if self.end_dim >= 0 else len(shape) + self.end_dim

        # Get the shapes to the left / right of flattened dims, as well as size of flattened middle
        shape_left = shape[:start_dim]
        shape_right = shape[end_dim + 1 :]
        shape_middle = t.prod(t.tensor(shape[start_dim : end_dim + 1])).item()

        return t.reshape(input, shape_left + (shape_middle,) + shape_right)

    def extra_repr(self) -> str:
        return ", ".join([f"{key}={getattr(self, key)}" for key in ["start_dim", "end_dim"]])

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self, d_in: int = 28**2, d_hidden: int = 100, d_out: int = 10):
        super().__init__()
        self.flatten = Flatten()
        self.layers = nn.Sequential(
            Linear(d_in, d_hidden),
            ReLU(),
            Linear(d_hidden, d_out)
        )

    def forward(self, x: Tensor) -> Tensor:
        input = self.flatten(x)
        output = self.layers(input)
        return output


tests.test_mlp_module(SimpleMLP)
tests.test_mlp_forward(SimpleMLP)

ModuleNotFoundError: No module named 'rich'

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self, d_in: int = 28**2, d_hidden: int = 100, d_out: int = 10):
        super().__init__()
        self.flatten = Flatten()
        self.layers = nn.Sequential(
            Linear(d_in, d_hidden),
            ReLU(),
            Linear(d_hidden, d_out)
        )

    def forward(self, x: Tensor) -> Tensor:
        input = self.flatten(x)
        output = self.layers(input)
        return output


tests.test_mlp_module(SimpleMLP)
tests.test_mlp_forward(SimpleMLP)

ModuleNotFoundError: No module named 'torchvision'

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self, d_in: int = 28**2, d_hidden: int = 100, d_out: int = 10):
        super().__init__()
        self.flatten = Flatten()
        self.layers = nn.Sequential(
            Linear(d_in, d_hidden),
            ReLU(),
            Linear(d_hidden, d_out)
        )

    def forward(self, x: Tensor) -> Tensor:
        input = self.flatten(x)
        output = self.layers(input)
        return output


tests.test_mlp_module(SimpleMLP)
tests.test_mlp_forward(SimpleMLP)

AssertionError: Your SimpleMLP should declare the following parameters in order.
Expected: ['linear1.weight', 'linear1.bias', 'linear2.weight', 'linear2.bias']
Actual: ['layers.0.weight', 'layers.0.bias', 'layers.2.weight', 'layers.2.bias']

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self, d_in: int = 28**2, d_hidden: int = 100, d_out: int = 10):
        super().__init__()
        self.flatten = Flatten()
        #self.layers = nn.Sequential(
        #    Linear(d_in, d_hidden),
        #    ReLU(),
        #    Linear(d_hidden, d_out)
        #)
        self.linear1 = Linear(d_in, d_hidden)
        self.relu = ReLU()
        self.linear2 = Linear(d_hidden, d_out)

    def forward(self, x: Tensor) -> Tensor:
        input = self.flatten(x)
        l1 = self.linear1(input)
        r1 = self.relu(l1)
        output = self.linear2(r1)
        return output


tests.test_mlp_module(SimpleMLP)
tests.test_mlp_forward(SimpleMLP)

All tests in `test_mlp_module` passed!
All tests in `test_mlp_forward` passed!


In [ ]:
word = "hello!"
pbar = tqdm(enumerate(word), total=len(word))
t0 = time.time()

for i, letter in pbar:
    time.sleep(1.0)
    pbar.set_postfix(i=i, letter=letter, time=f"{time.time()-t0:.3f}")

100%|██████████| 6/6 [00:06<00:00,  1.01s/it, i=5, letter=!, time=6.034]


In [ ]:
device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")
print(device)

mps


In [ ]:
MNIST_TRANSFORM = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize(0.1307, 0.3081),
    ]
)

def get_mnist(trainset_size: int = 10_000, testset_size: int = 1_000) -> tuple[Subset, Subset]:
    """Returns a subset of MNIST training data."""

    # Get original datasets, which are downloaded to "./data" for future use
    mnist_trainset = datasets.MNIST(exercises_dir / "data", train=True, download=True, transform=MNIST_TRANSFORM)
    mnist_testset = datasets.MNIST(exercises_dir / "data", train=False, download=True, transform=MNIST_TRANSFORM)

    # # Return a subset of the original datasets
    mnist_trainset = Subset(mnist_trainset, indices=range(trainset_size))
    mnist_testset = Subset(mnist_testset, indices=range(testset_size))

    return mnist_trainset, mnist_testset


mnist_trainset, mnist_testset = get_mnist()
mnist_trainloader = DataLoader(mnist_trainset, batch_size=64, shuffle=True)
mnist_testloader = DataLoader(mnist_testset, batch_size=64, shuffle=False)

# Get the first batch of test data, by starting to iterate over `mnist_testloader`
for img_batch, label_batch in mnist_testloader:
    print(f"{img_batch.shape=}\n{label_batch.shape=}\n")
    break

# Get the first datapoint in the test set, by starting to iterate over `mnist_testset`
for img, label in mnist_testset:
    print(f"{img.shape=}\n{label=}\n")
    break

t.testing.assert_close(img, img_batch[0])
assert label == label_batch[0].item()

100%|██████████| 9.91M/9.91M [00:00<00:00, 13.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 3.63MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 13.3MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.09MB/s]


img_batch.shape=torch.Size([64, 1, 28, 28])
label_batch.shape=torch.Size([64])

img.shape=torch.Size([1, 28, 28])
label=7



In [ ]:
model = SimpleMLP().to(device)

batch_size = 64
epochs = 3

mnist_trainset, _ = get_mnist()
mnist_trainloader = DataLoader(mnist_trainset, batch_size=batch_size, shuffle=True)

optimizer = t.optim.Adam(model.parameters(), lr=1e-3)
loss_list = []

for epoch in range(epochs):
    pbar = tqdm(mnist_trainloader)

    for imgs, labels in pbar:
        # Move data to device, perform forward pass
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)

        # Calculate loss, perform backward pass
        loss = F.cross_entropy(logits, labels)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        # Update logs & progress bar
        loss_list.append(loss.item())
        pbar.set_postfix(epoch=f"{epoch + 1}/{epochs}", loss=f"{loss:.3f}")

line(
    loss_list,
    x_max=epochs * len(mnist_trainset),
    labels={"x": "Examples seen", "y": "Cross entropy loss"},
    title="SimpleMLP training on MNIST",
    width=500,
)

100%|██████████| 157/157 [00:01<00:00, 147.21it/s, epoch=3/3, loss=0.221]


In [ ]:
@dataclass
class SimpleMLPTrainingArgs:
    """
    Defining this class implicitly creates an __init__ method, which sets arguments as below, e.g.
    self.batch_size=64. Any of these fields can also be overridden when you create an instance, e.g.
    SimpleMLPTrainingArgs(batch_size=128).
    """

    batch_size: int = 64
    epochs: int = 3
    learning_rate: float = 1e-3


def train(args: SimpleMLPTrainingArgs) -> tuple[list[float], SimpleMLP]:
    """
    Trains & returns the model, using training parameters from the `args` object. Returns the model,
    and loss list.
    """
    model = SimpleMLP().to(device)

    mnist_trainset, _ = get_mnist()
    mnist_trainloader = DataLoader(mnist_trainset, batch_size=args.batch_size, shuffle=True)

    optimizer = t.optim.Adam(model.parameters(), lr=args.learning_rate)
    loss_list = []

    for epoch in range(args.epochs):
        pbar = tqdm(mnist_trainloader)

        for imgs, labels in pbar:
            # Move data to device, perform forward pass
            imgs, labels = imgs.to(device), labels.to(device)
            logits = model(imgs)

            # Calculate loss, perform backward pass
            loss = F.cross_entropy(logits, labels)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            # Update logs & progress bar
            loss_list.append(loss.item())
            pbar.set_postfix(epoch=f"{epoch + 1}/{args.epochs}", loss=f"{loss:.3f}")

    return loss_list, model


args = SimpleMLPTrainingArgs()
loss_list, model = train(args)

line(
    loss_list,
    x_max=args.epochs * len(mnist_trainset),
    labels={"x": "Examples seen", "y": "Cross entropy loss"},
    title="SimpleMLP training on MNIST",
    width=600,
)

100%|██████████| 157/157 [00:01<00:00, 141.11it/s, epoch=3/3, loss=0.443]


In [ ]:
def train(args: SimpleMLPTrainingArgs) -> tuple[list[float], list[float], SimpleMLP]:
    """
    Trains the model, using training parameters from the `args` object.

    Returns:
        The model, and lists of loss & accuracy.
    """
    
    model = SimpleMLP().to(device)

    mnist_trainset, mnist_testset = get_mnist()
    mnist_trainloader = DataLoader(mnist_trainset, batch_size=args.batch_size, shuffle=True)
    mnist_testloader = DataLoader(mnist_testset, batch_size=args.batch_size, shuffle=False)

    optimizer = t.optim.Adam(model.parameters(), lr=args.learning_rate)
    loss_list = []
    accuracy_list = []

    for epoch in range(args.epochs):
        
        # Training loop
        pbar = tqdm(mnist_trainloader)
        model.train()
        for imgs, labels in pbar:
            # Move data to device, perform forward pass
            imgs, labels = imgs.to(device), labels.to(device)
            logits = model(imgs)

            # Calculate loss, perform backward pass
            loss = F.cross_entropy(logits, labels)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            # Update logs & progress bar
            loss_list.append(loss.item())
            pbar.set_postfix(epoch=f"{epoch + 1}/{args.epochs}", loss=f"{loss:.3f}")

        # Validation loop
        model.eval()
        correct, total = 0, 0
        with t.inference_mode():
            for imgs, labels in mnist_testloader:
                imgs, labels = imgs.to(device), labels.to(device)
                logits = model(imgs)
                predictions = t.argmax(logits, dim=-1)
                correct += (predictions == labels).sum().item()
                total += labels.shape[0]
            accuracy_list.append(correct / total)

    return loss_list, accuracy_list, model


args = SimpleMLPTrainingArgs()
loss_list, accuracy_list, model = train(args)

line(
    y=[loss_list, [0.1] + accuracy_list],  # we start by assuming a uniform accuracy of 10%
    use_secondary_yaxis=True,
    x_max=args.epochs * len(mnist_trainset),
    labels={"x": "Num examples seen", "y1": "Cross entropy loss", "y2": "Test Accuracy"},
    title="SimpleMLP training on MNIST",
    width=600,
)

100%|██████████| 157/157 [00:01<00:00, 137.42it/s, epoch=3/3, loss=0.131]
